# EcoBuild | Reproducible Building Energy Prediction and Decision-Support Analysis

**Individual research notebook — Veronica Broadway**  
**Project:** EcoBuild, Building Energy Efficiency Predictor  
**Research focus:** data quality, exploratory analytics, predictive model evaluation, reliability across building configurations, and responsible decision support.

## Study roadmap
1. Document data provenance, methods, and environmental settings.
2. Audit the dataset and explore building design relationships.
3. Create a reproducible held-out evaluation with **both** heating load (Y1) and cooling load (Y2).
4. Compare a training-mean baseline, linear regression, random forest, and gradient boosting; select using training-only cross-validation.
5. Evaluate previously unseen holdout rows, analyze error by height, and inspect feature importance.
6. Compare ordinary versus geometry-grouped cross-validation, reproduce the exact 192-design grouping from Namita’s work, and quantify the effect using repeated holdouts.
7. Check subgroup reliability on new designs, attribute team SHAP and app-model contributions accurately, and export research-ready figures/tables.

**Scope statement:** This analysis evaluates predictions against **simulation-generated labels** from the UCI Energy Efficiency dataset. It does not establish performance on occupied buildings, cost savings, causal effects of design changes, or demographic fairness. Results printed here are generated when the notebook runs; narrative text is not a substitute for measured outputs.

**Research question (feasible for the current dataset):** How accurately do conventional and nonlinear regression models predict simulated heating and cooling loads from early-stage building characteristics, and how consistently do prediction errors vary across building-height configurations?

**Accessibility conventions:** Headings are hierarchical; no result relies on color alone. Figures include descriptive titles, explicit axis labels, readable text, and adjacent text alternatives. All essential chart results are also available in data tables. Colors are chosen for contrast on white; descriptions should be updated if figures change.

## 1. Reproducibility and project setup
This notebook is intended for **Google Colab** or a local Python installation with internet access for UCI retrieval. The only remote dependency is the public UCI dataset (ID 242). If retrieval fails, upload the authoritative ENB2012 Excel file; no synthetic replacement data are generated.

**Methodological decision:** Set the random seed, record software versions, and export all numerical results so claims in the paper can be traced to a run.

In [ ]:
# Lightweight setup: only install if missing.
import importlib.util, subprocess, sys
if importlib.util.find_spec("ucimlrepo") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "ucimlrepo"])

import os, json, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn import __version__ as sklearn_version

SEED = 42
OUT = Path("ecobuild_research_outputs")
OUT.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 11, "axes.titlesize": 13, "axes.labelsize": 11,
                     "savefig.dpi": 250, "figure.dpi": 110})
FEATURES = [f"X{i}" for i in range(1, 9)]
TARGETS = ["Y1", "Y2"]
NAMES = {"X1":"Relative compactness", "X2":"Surface area", "X3":"Wall area",
         "X4":"Roof area", "X5":"Overall height", "X6":"Orientation",
         "X7":"Glazing area", "X8":"Glazing area distribution",
         "Y1":"Heating load", "Y2":"Cooling load"}
VERSIONS = {"python": platform.python_version(), "pandas": pd.__version__,
            "numpy": np.__version__, "scikit_learn": sklearn_version}
print("Software:", VERSIONS, "\nOutput folder:", OUT.resolve())

## 2. Dataset source, variable dictionary, and governance
**Source:** UCI Machine Learning Repository, *Energy Efficiency* dataset (ID 242), originally associated with Tsanas and Xifara (2012). The 768 rows represent simulated building configurations, not measured buildings. There are eight inputs (X1–X8) and two numerical outcomes (Y1 heating load; Y2 cooling load).

| Field | Description | Interpretation |
|---|---|---|
| X1 | Relative compactness | Design geometry |
| X2 | Surface area | Building envelope geometry |
| X3 | Wall area | Building envelope geometry |
| X4 | Roof area | Building envelope geometry |
| X5 | Overall height | Two design categories, 3.5 m and 7.0 m |
| X6 | Orientation | Encoded design direction; categorical, not a continuous physical scale |
| X7 | Glazing area | Fractional design variable |
| X8 | Glazing area distribution | Encoded glazing arrangement; categorical |
| Y1 | Heating load | Regression target |
| Y2 | Cooling load | Regression target |

**Governance:** Public simulation data contain no person-level attributes. Keep attribution and dataset citation in the paper, record preprocessing decisions, do not infer operational energy consumption or environmental impact without field validation.

**Reproducibility:** The following cell attempts UCI retrieval, then looks for a local Excel/CSV file. A user-supplied file is used only if the expected columns exist.

In [ ]:
DATA_SOURCE = None
try:
    from ucimlrepo import fetch_ucirepo
    repo = fetch_ucirepo(id=242)
    raw = pd.concat([repo.data.features, repo.data.targets], axis=1)
    DATA_SOURCE = "UCI Energy Efficiency repository, dataset 242 (ucimlrepo)"
except Exception as exc:
    print("UCI fetch unavailable:", str(exc)[:250])
    candidates = [Path("EcoBuild Data - ENB2012_data.xlsx"), Path("ENB2012_data.xlsx"),
                  Path("ENB2012_data.csv"), Path("/content/ENB2012_data.xlsx")]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        raise FileNotFoundError("Upload ENB2012_data.xlsx (or EcoBuild Data - ENB2012_data.xlsx) "
                                "and rerun this cell; no substitute data will be used.") from exc
    raw = pd.read_csv(path) if path.suffix.lower() == ".csv" else pd.read_excel(path)
    DATA_SOURCE = str(path)

# Normalize column names if the repository returns full feature labels.
raw.columns = [str(c).strip() for c in raw.columns]
if not set(FEATURES + TARGETS).issubset(raw.columns):
    reverse = {"Relative Compactness":"X1", "Surface Area":"X2", "Wall Area":"X3",
               "Roof Area":"X4", "Overall Height":"X5", "Orientation":"X6",
               "Glazing Area":"X7", "Glazing Area Distribution":"X8",
               "Heating Load":"Y1", "Cooling Load":"Y2"}
    raw = raw.rename(columns=reverse)
missing = sorted(set(FEATURES + TARGETS) - set(raw.columns))
if missing: raise ValueError(f"Missing expected variables: {missing}; got {list(raw.columns)}")
df = raw[FEATURES + TARGETS].copy()
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")
print("Source:", DATA_SOURCE)
print("Dataset shape:", df.shape)
display(df.head().rename(columns=NAMES))

## 3. Data quality and structural audit
Check size, numeric types, missingness, duplicates, unique values, and ranges **before** modeling. Duplicate features with conflicting targets would need investigation; rows representing valid repeated configurations should not be removed automatically. Summarize exactly what is discovered in the paper's Data section.

**Accessibility:** Printed tables have explicit column names and do not encode quality solely with colors.

In [ ]:
quality = pd.DataFrame({
    "variable": FEATURES + TARGETS,
    "meaning": [NAMES[c] for c in FEATURES + TARGETS],
    "dtype": [str(df[c].dtype) for c in FEATURES + TARGETS],
    "missing": [int(df[c].isna().sum()) for c in FEATURES + TARGETS],
    "unique_values": [int(df[c].nunique()) for c in FEATURES + TARGETS],
    "minimum": [df[c].min() for c in FEATURES + TARGETS],
    "maximum": [df[c].max() for c in FEATURES + TARGETS],
})
display(quality)
print("Rows:", len(df), "| Full duplicate rows:", int(df.duplicated().sum()),
      "| Duplicate input rows:", int(df.duplicated(FEATURES).sum()))
print("Conflicting outputs for identical input configurations:",
      int((df.groupby(FEATURES, dropna=False)[TARGETS].nunique(dropna=False) > 1).any(axis=1).sum()))
quality.to_csv(OUT / "table_01_data_quality.csv", index=False)
if df.isna().any().any():
    raise ValueError("Missing or nonnumeric values found. Inspect source; decide how to handle before training.")
if len(df) != 768: warnings.warn("Dataset row count differs from the expected 768. Verify the source.")
if not set(df["X5"].unique()).issubset({3.5, 7.0}):
    warnings.warn("Unexpected height values: update the height-group audit before running it.")

## 4. Exploratory data analysis (EDA): central tendencies and design groups
EDA characterizes the sample **without treating associations as causal**. Evaluate target distributions, height-group means and medians, and design variable relationships. These summaries are descriptive observations, not trained model performance.

**Questions:** Do the two targets vary broadly across configurations? How do the height groups differ? Are within-height correlations different from pooled correlations?

In [ ]:
summary = df[TARGETS].agg(["count", "mean", "std", "min", "median", "max"]).T
summary.index = [NAMES[c] for c in summary.index]
height_summary = df.groupby("X5", observed=True).agg(
    n=("Y1", "size"), heating_mean=("Y1", "mean"), heating_median=("Y1", "median"),
    heating_min=("Y1", "min"), heating_max=("Y1", "max"),
    cooling_mean=("Y2", "mean"), cooling_median=("Y2", "median"))
display(summary.round(3))
display(height_summary.round(3))
print("Overall heating/cooling correlation:", round(df["Y1"].corr(df["Y2"]), 4))
for h, g in df.groupby("X5"):
    print(f"At height {h:g} m: compactness/heating correlation {g['X1'].corr(g['Y1']):.3f}")
summary.to_csv(OUT / "table_02_target_descriptives.csv")
height_summary.to_csv(OUT / "table_03_height_group_descriptives.csv")

### Figure 1. Heating and cooling load distributions
Two separate histograms preserve axis clarity and avoid comparing different counts on one ambiguous plot.

**Figure text alternative (update after executing):** The two distributions show the number of simulated configurations at different heating-load and cooling-load values. Vertical dashed lines identify their means. Read the accompanying summary table for exact means, minima, and maxima. The spread indicates variability in simulated loads, not measured energy use.

In [ ]:
for target in TARGETS:
    fig, ax = plt.subplots(figsize=(8, 4.6))
    ax.hist(df[target], bins=24, color="#286090", edgecolor="white", linewidth=.8)
    avg = df[target].mean()
    ax.axvline(avg, color="#963F00", linestyle="--", linewidth=2.2,
               label=f"Mean = {avg:.2f}")
    ax.set(title=f"Distribution of simulated {NAMES[target].lower()} across building designs",
           xlabel=f"{NAMES[target]} (dataset units)", ylabel="Number of configurations")
    ax.legend(loc="upper right")
    fig.tight_layout()
    fig.savefig(OUT / f"figure_01_{target}_distribution.png", bbox_inches="tight")
    plt.show()
    print(f"Accessible summary, {target}: n={len(df)}, min={df[target].min():.2f}, "
          f"mean={avg:.2f}, median={df[target].median():.2f}, max={df[target].max():.2f}.")